# 2 · The naive baseline · `full_realtime`

<img src="https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120" alt="Redis"/>

<a href="https://colab.research.google.com/github/redis-field-engineering/redis-dsp-demo/blob/main/notebooks/02_full_realtime_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part of the Redis DSP candidate-generation demo. The full sequence walks the bid path from naive to fast across six notebooks; this is `02_full_realtime_baseline.ipynb`.

**To run in Colab:** click the badge above, then *Runtime → Run all*. The setup cells below clone the repo, install dependencies, start a Redis Stack server, and load the synthetic dataset.

**To run locally:** make sure the docker-compose stack is up (`make up` from the repo root). The setup cells detect a local environment and skip the Colab-specific steps.

## Setup

These five cells prepare the environment. They are idempotent — safe to re-run, safe in either Colab or local. On Colab the first run takes about 60–90 seconds (pip install + apt install + dataset generation). Subsequent runs are near-instant because everything is cached.

In [1]:
# Setup 1/5 · clone the repo (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("pyproject.toml"):
    print("Cloning https://github.com/redis-field-engineering/redis-dsp-demo ...")
    os.system("git clone -q https://github.com/redis-field-engineering/redis-dsp-demo.git _repo")
    os.system("cp -R _repo/. ./")
    os.system("rm -rf _repo")
    print("Repo cloned.")
elif not IN_COLAB:
    print("Local environment detected — skipping clone.")
else:
    print("Repo already present.")

Local environment detected — skipping clone.


In [2]:
# Setup 2/5 · install Python dependencies (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        f'{sys.executable} -m pip install -q '
        '"redis[hiredis]>=5.2.0" "pydantic>=2.9.0" "pandas>=2.2.0" "pyarrow>=18.0.0"'
    )
    print("Dependencies installed.")
else:
    print("Local environment detected — skipping pip install (assumes deps are already installed).")

Local environment detected — skipping pip install (assumes deps are already installed).


In [3]:
# Setup 3/5 · install and start Redis Stack (Colab only).
import os, sys, shutil
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if shutil.which("redis-stack-server") is None:
        print("Installing redis-stack-server ...")
        os.system(
            'curl -fsSL https://packages.redis.io/gpg | '
            'sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg'
        )
        os.system(
            'echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] '
            'https://packages.redis.io/deb $(lsb_release -cs) main" '
            '| sudo tee /etc/apt/sources.list.d/redis.list > /dev/null'
        )
        os.system("sudo apt-get update -qq > /dev/null 2>&1")
        os.system("sudo apt-get install -qq -y redis-stack-server > /dev/null 2>&1")
    os.system("redis-stack-server --daemonize yes > /dev/null 2>&1")
    print("redis-stack-server started on :6379")
else:
    print("Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).")

Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).


In [4]:
# Setup 4/5 · choose the Redis URL.
import os, sys
IN_COLAB = "google.colab" in sys.modules
default_port = "6379" if IN_COLAB else "6381"
REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", default_port)
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")
auth = f":{REDIS_PASSWORD}@" if REDIS_PASSWORD else ""
REDIS_URL = f"redis://{auth}{REDIS_HOST}:{REDIS_PORT}/0"
os.environ["DEMO_REDIS_URL"] = REDIS_URL
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://localhost:6381/0


In [5]:
# Setup 5/5 · generate and load the synthetic dataset (only if Redis is empty).
import sys, subprocess
from pathlib import Path

# Find the repo root so we can run `python -m data.synthetic` reliably.
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from redis import Redis
client = Redis.from_url(REDIS_URL, decode_responses=True)
if not client.ping():
    raise RuntimeError(f"Redis at {REDIS_URL} did not answer PING")

if client.exists("meta:dataset_loaded"):
    print(
        f"Dataset already loaded: "
        f"{client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )
else:
    print("Generating synthetic dataset (~30 seconds) ...")
    subprocess.run(
        [sys.executable, "-m", "data.synthetic",
         "--output", "data/generated/synthetic",
         "--num-users", "4000",
         "--num-campaigns", "2500",
         "--num-interactions", "120000",
         "--feature-count", "12"],
        cwd=_repo_root, check=True,
    )
    print("Loading dataset into Redis ...")
    subprocess.run(
        [sys.executable, "-m", "data.load_redis",
         "--redis-url", REDIS_URL,
         "--dataset-dir", "data/generated/synthetic"],
        cwd=_repo_root, check=True,
    )
    print(
        f"Done. {client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )

Dataset already loaded: 4000 users, 2500 campaigns.


## Walkthrough

From here on the notebook is the demonstration.

In [6]:
# Locate the repo root so `notebooks._demo_setup` is importable regardless
# of where the kernel was launched (the package layout requires the repo
# root on sys.path).
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from notebooks._demo_setup import connect_redis, StepTimer
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## Step-by-step

The bid engine walks the request through the steps in order. The timer
captures the wall-clock cost of each.


In [7]:
IDENTITY_TOKEN = 'id_00042_01'
timer = StepTimer()

with timer.step('identity_resolution'):
    maid_id = client.get(f'identity:{IDENTITY_TOKEN}')

with timer.step('profile_fetch'):
    profile = client.hgetall(f'maid:{maid_id}')

with timer.step('campaign_id_scan'):
    # full_realtime materializes the entire campaign universe.
    campaign_ids = sorted(
        key.removeprefix('campaign:')
        for key in client.scan_iter(match='campaign:*', count=2000)
    )

with timer.step('campaign_fetch_pipelined'):
    pipe = client.pipeline(transaction=False)
    for cid in campaign_ids:
        pipe.hgetall(f'campaign:{cid}')
    payloads = pipe.execute()

print(f'maid_id        = {maid_id}')
print(f'profile fields = {len(profile)}')
print(f'campaigns seen = {len(campaign_ids)}')
print()
print(timer.summary())

maid_id        = maid_00042
profile fields = 14
campaigns seen = 2500

             identity_resolution    0.313 ms
                   profile_fetch    0.368 ms
                campaign_id_scan   15.380 ms
        campaign_fetch_pipelined  114.756 ms
--------------------------------------------
                           TOTAL  130.817 ms


## Filter + rerank in app memory

The prototype's `filter_campaigns_for_user` function holds the same
eligibility logic for every mode — geo, state, device, card tier, segment
required/any_of/none_of, pacing, budget, frequency, **and** the float-score
`taxonomy_filter`. Running it over all 2500 campaigns is the work that
`full_realtime` is paying for.


In [8]:
from app.candidate import filter_campaigns_for_user
from app.models import Campaign, UserProfile
from app.ranking import rerank_campaigns

# Re-hydrate the profile + campaigns into prototype model instances.
user = UserProfile.from_redis_hash(profile)
campaigns = [Campaign.from_redis_hash(p) for p in payloads if p]

with timer.step('filter_in_app'):
    eligible = filter_campaigns_for_user(user, campaigns)

with timer.step('rerank'):
    top_5 = rerank_campaigns(user, eligible, top_k=5)

print(f'filtered  {len(campaigns)} -> {len(eligible)} eligible')
print('top 5 ranked:')
for ranked in top_5:
    print(f'  {ranked.campaign_id}  score={ranked.score:.4f}')
print()
print(timer.summary())

filtered  2500 -> 10 eligible
top 5 ranked:
  c00848  score=4.8107
  c01551  score=4.4065
  c01222  score=4.2812
  c02229  score=4.1223
  c01617  score=3.6611

             identity_resolution    0.313 ms
                   profile_fetch    0.368 ms
                campaign_id_scan   15.380 ms
        campaign_fetch_pipelined  114.756 ms
                   filter_in_app    1.561 ms
                          rerank    0.083 ms
--------------------------------------------
                           TOTAL  132.462 ms


## What the timing tells us

A few things to point out on the call:

- **`campaign_fetch_pipelined`** dominates. Pulling 2500 hashes back is one
  pipelined round trip but a few thousand actual Redis ops, and the response
  payload is non-trivial.
- **`filter_in_app`** is also significant — every one of those 2500
  campaigns gets the full eligibility check.
- The decision-path total here is the number every other mode is trying
  to beat. The published headline runs the same path on a tuned VM and
  comes in around `~40 ms` p50. The remaining notebooks show how the
  prototype gets that down to single-digit milliseconds.

Note: this is `concurrency = 1` running on whatever shape the local docker
stack happens to be on. The relative ordering of modes is what matters
in this demo, not the absolute numbers — see `reports/benchmark_report.md`
for the tuned VM run.
